<div style="background-color:white; text-align:center; font-family:Arial, Helvetica, sans-serif; padding:50px;">
  <!-- Tytuł -->
  <div style="color:#993520; font-size:60px; font-weight:bold; margin-bottom:20px;">
    WEB SCRAPING AND SOCIAL MEDIA SCRAPING
  </div>

  <!-- Podtytuł -->
  <div style="color:#993520; font-size:35px; margin-bottom:40px;">
    Scraping with Selenium
  </div>

  <!-- Autor -->
  <div style="color:black; font-size:30px; margin-bottom:10px;">
    Maciej Świtała, PhD
  </div>
  <div style="color:black; font-size:30px; margin-bottom:40px;">
    Ewa Weychert, PhD in spe
  </div>

  <!-- Data / semestr -->
  <div style="color:black; font-size:30px; margin-bottom:40px;">
    Spring 2026
  </div>

  <!-- Logo -->
  <div>
    <img src="img/wne-logo-new-en.jpg" alt="WNE Logo" style="max-width:400px; height:auto;">
  </div>
</div>


### Setting up the environment

Let us (install if neccessary and) load the needed libraries.

In [19]:
!pip --quiet install pandas numpy selenium webdriver-manager fake-useragent


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# FOR DATA PROCESSING:
import pandas as pd
import numpy as np

# FOR MEASURING COMPUTATION TIME, CREATING FIXED DELAYS:
import time

# FOR APPLYING SELENIUM:
import selenium # Python Selenium
from selenium import webdriver # for specifying webdriver

from webdriver_manager.chrome import ChromeDriverManager # chromedriver for automatized access to Chrome

from selenium.webdriver.chrome.service import Service # needed since Selenium 4.10.0 see: https://github.com/SeleniumHQ/selenium/commit/9f5801c82fb3be3d5850707c46c3f8176e3ccd8e

from selenium.webdriver.support.ui import WebDriverWait # this three enable waiting until sth is displayed on website
from selenium.webdriver.support import expected_conditions as EC # for checking visibility of an element
from selenium.webdriver.common.by import By # for checking element visibility by XPath

# FOR SAVING DATA:
import pickle # pickle format of saved output

def save_object(obj, filename): #  function defined for saving Python objects
    with open(filename, 'wb') as output: # overwrites any existing file
        pickle.dump(obj, output, pickle.HIGHEST_PROTOCOL)

### Browsers' drivers

To proceed with Selenium, browser's webdriver need to be installed - which exactly depends on the browser we want to use. Here, let us one of the most popular browsers, i.e., Chrome.

In [7]:
chromepath = ChromeDriverManager().install(); print(chromepath)

C:\Users\Admin\.wdm\drivers\chromedriver\win64\145.0.7632.117\chromedriver-win32/chromedriver.exe


### Example: Scrapping news on climate solutions from CNN

Before we start: can we scrap news from the CNN website?

We should investigate the robots.txt file first, see: https://edition.cnn.com/robots.txt. Looks like one can scrap some of the website's subsections.

What about the website's terms of service? Let us investigate it, see: https://edition.cnn.com/2015/01/06/world/terms-service/index.html. No specific statements on webscraping. Yet, clearly we must not collect and process personal data.

It is highly unlikely that we could infringe anyone's copyright by scraping content from this website. Also, it is hard to imagine causing a material damage unless we break the website.

Conclusion: we can scrap it as long as it is consistent with robots.txt and it is not personal data.

Step 1: Open the CNN website with Selenium.

In [8]:
website = "https://edition.cnn.com/climate/solutions/page/1"

service = Service(executable_path = chromepath) 
options = webdriver.ChromeOptions()
driver = webdriver.Chrome(service = service, options = options) # opens the browser

driver.maximize_window() # maximizes browser's window
driver.get(website) # opens a website

Step 2: Accept the cookies.

In [9]:
website = "https://edition.cnn.com/climate/solutions/page/1"

service = Service(executable_path = chromepath) 
options = webdriver.ChromeOptions()
driver = webdriver.Chrome(service = service, options = options) # opens the browser

driver.maximize_window() # maximizes browser's window
driver.get(website) # opens a website

cookies_button_xpath = '''//*[@id="onetrust-accept-btn-handler"]'''

# wait at most 30 seconds until cookies button is visible
WebDriverWait(driver, 30).until(EC.visibility_of_element_located((By.XPATH, cookies_button_xpath))) 

# + wait random time drawn from specific (strongly right-side-skewed) distribution to better imitate human behavior
time.sleep(np.random.chisquare(1)+3)

content = driver.find_element("xpath", cookies_button_xpath) # finds the button
content.click() # clicks the button

Step 3: Build a scraper collecting links to subpages with single news.

In [11]:
start = time.time()

website = "https://edition.cnn.com/climate/solutions/page/1"

service = Service(executable_path = chromepath) 
options = webdriver.ChromeOptions()
driver = webdriver.Chrome(service = service, options = options) # opens the browser

driver.maximize_window() # maximizes browser's window
driver.get(website) # opens a website

cookies_button_xpath = '''//*[@id="onetrust-accept-btn-handler"]'''

# wait at most 30 seconds until cookies button is visible
WebDriverWait(driver, 30).until(EC.visibility_of_element_located((By.XPATH, cookies_button_xpath))) 

# + wait random time drawn from specific (strongly right-side-skewed) distribution to better imitate human behavior
time.sleep(np.random.chisquare(1)+3)

content = driver.find_element("xpath", cookies_button_xpath) # finds the button
content.click() # clicks the button
time.sleep(3) # extra time needed for the button to disappear

# wait at most 30 seconds until webpage is reloaded
WebDriverWait(driver, 30).until(EC.visibility_of_element_located((By.XPATH, '''//a[contains(@data-link-type,'article')]'''))) 

# + wait random time drawn from specific (strongly right-side-skewed) distribution to better imitate human behavior
time.sleep(np.random.chisquare(1)+3)

# finds all <a> elements on website that include: data-link-type='article'
tags = driver.find_elements('xpath','''//a[contains(@data-link-type,'article')]''')

# finds all links from the 'tags'
hrefs = []
for tag in tags:
    href = tag.get_attribute('href') # for each <a> finds 'href', i.e., link to subpage
    if(href not in hrefs): # here we handle duplicates
        hrefs.append(href)

driver.close() # this closes the webdriver
        
print(len(hrefs))

end = time.time()
print(end-start)

29
29.14797854423523


Step 4: Scale the procedure of collecting the links to subpages for the whole section of the website.

In [13]:
start = time.time()

service = Service(executable_path = chromepath) 
options = webdriver.ChromeOptions()
driver = webdriver.Chrome(service = service, options = options) # opens the browser

driver.maximize_window() # maximizes browser's window
website = "https://edition.cnn.com/climate/solutions/page/1"

driver.get(website) # opens a website

cookies_button_xpath = '''//*[@id="onetrust-accept-btn-handler"]'''

# wait at most 30 seconds until cookies button is visible
WebDriverWait(driver, 30).until(EC.visibility_of_element_located((By.XPATH, cookies_button_xpath))) 

# + wait random time drawn from specific (strongly right-side-skewed) distribution to better imitate human behavior
time.sleep(np.random.chisquare(1)+3)

content = driver.find_element("xpath", cookies_button_xpath) # finds the button
content.click() # clicks the button
time.sleep(3) # extra time needed for the button to disappear

hrefs = []

for i in range(1,14): # looks like there are 13 pages (state for 06.03.2025, 16:36)

    try:

        # wait at most 30 seconds until webpage is reloaded
        WebDriverWait(driver, 30).until(EC.visibility_of_element_located((By.XPATH, '''//a[contains(@data-link-type,'article')]'''))) 
        
        # + wait random time drawn from specific (strongly right-side-skewed) distribution to better imitate human behavior
        time.sleep(np.random.chisquare(1)+3)

        # finds all <a> elements on website that include: data-link-type='article'
        tags = driver.find_elements('xpath','''//a[contains(@data-link-type,'article')]''')

        # finds all links from the 'tags'
        for tag in tags:
            href = tag.get_attribute('href') # for each <a> finds 'href', i.e., link to subpage
            if(href not in hrefs): # here we handle duplicates
                hrefs.append(href)

        website = "https://edition.cnn.com/climate/solutions/page/"+str(i)
        driver.get(website) # opens a website

    except:
        continue
    
driver.close() # this closes the webdriver
    
print(len(hrefs))

end = time.time()
print(end-start)

174
190.21959900856018


Step 5: Access the collected links and collect the data.

CAUTION! It appears we do not have enough time to scrap everything. Let us collect first 30 texts.

In [14]:
start = time.time()

service = Service(executable_path = chromepath) 
options = webdriver.ChromeOptions()
driver = webdriver.Chrome(service = service, options = options) # opens the browser

driver.maximize_window() # maximizes browser's window
website = hrefs[0]

driver.get(website) # opens a website

cookies_button_xpath = '''//*[@id="onetrust-accept-btn-handler"]'''

# wait at most 30 seconds until cookies button is visible
WebDriverWait(driver, 30).until(EC.visibility_of_element_located((By.XPATH, cookies_button_xpath))) 

# + wait random time drawn from specific (strongly right-side-skewed) distribution to better imitate human behavior
time.sleep(np.random.chisquare(1)+3)

content = driver.find_element("xpath", cookies_button_xpath) # finds the button
content.click() # clicks the button
time.sleep(3) # extra time needed for the button to disappear

texts = []

for href in hrefs[1:30]: # here we limit to first 30 articles

    try:

        article_content_xpath = '''//div[@class='article__content']'''
        
        # wait at most 30 seconds until the article is visible
        WebDriverWait(driver, 30).until(EC.visibility_of_element_located((By.XPATH, article_content_xpath))) 
        
        # + wait random time drawn from specific (strongly right-side-skewed) distribution to better imitate human behavior
        time.sleep(np.random.chisquare(1)+3)

        content = driver.find_element("xpath", article_content_xpath) # finds the content
        texts.append(content.text)
        
        driver.get(href)
    
    except:
        continue
    
driver.close() # this closes the webdriver
    
print(len(texts))

end = time.time()
print(end-start)

29
331.498841047287


In [15]:
# here, we save what we scraped as pickle or txt
# obviously we could use some different formats though
save_object(texts, r'outputs/output_CNNclimate.pkl')
save_object(texts, r'outputs/output_CNNclimate.txt')

### Example: setting up more Selenium options

Selenium offers plenty of options that can be useful in different context. Let us focus on those that appear the most popular.

In [16]:
start = time.time()

website = "https://edition.cnn.com/climate/solutions/page/1"

service = Service(executable_path = chromepath) 
options = webdriver.ChromeOptions()

############################## CHECK OUT THE OPTIONS ADDED ##############################
options.add_argument("--start-maximized") # the window will start as maximized
options.add_argument("--headless") # browser's window will not be displayed when this applied
options.add_argument("--incognito") # incognito mode
options.add_argument("--no-sandbox")
# sandboxing is a security feature that helps isolate web pages in their own secure
# "sandbox" to prevent malicious code from affecting the entire system
options.add_argument("--disable-gpu")
# instructs Chrome to disable the GPU (Graphics Processing Unit) hardware acceleration
options.add_argument("--disable-notifications")
options.add_argument("--disable-infobars")
options.add_argument("--disable-extensions")
options.add_argument("--disable-web-security")
#########################################################################################

driver = webdriver.Firefox(service = service, options = options) # opens the browser

driver.maximize_window() # maximizes browser's window
driver.get(website) # opens a website

cookies_button_xpath = '''//*[@id="onetrust-accept-btn-handler"]''' 

# wait at most 30 seconds until cookies button is visible
WebDriverWait(driver, 30).until(EC.visibility_of_element_located((By.XPATH, cookies_button_xpath))) 
# + wait random time drawn from specific (strongly right-side-skewed) distribution to better imitate human behavior
time.sleep(np.random.chisquare(1)+3)

content = driver.find_element("xpath", cookies_button_xpath) # finds the button
content.click() # clicks the button

for i in range(1,14): # there are 13 pages, state as for 06.03.2025, 16:36

    # wait at most 30 seconds until webpage is reloaded
    WebDriverWait(driver, 30).until(EC.visibility_of_element_located((By.XPATH, '''//a[contains(@data-link-type,'article')]'''))) 
    # + wait random time drawn from specific (strongly right-side-skewed) distribution to better imitate human behavior
    time.sleep(np.random.chisquare(1)+3)

    # finds all <a> elements on website that include: data-link-type='article'
    tags = driver.find_elements('xpath','''//a[contains(@data-link-type,'article')]''')

    # finds all links from the 'tags'
    for tag in tags:
        href = tag.get_attribute('href') # for each <a> finds 'href', i.e., link to subpage
        if(href not in hrefs): # here we handle duplicates
            hrefs.append(href)

    website = "https://edition.cnn.com/climate/solutions/page/"+str(i)
    driver.get(website) # opens a website
        
texts = []

for href in hrefs[0:30]: # here we limit to first 30 articles
    
    driver.get(href)

    article_content_xpath = '''//div[@class='article__content']'''
    
    # wait at most 30 seconds until the article is visible
    WebDriverWait(driver, 30).until(EC.visibility_of_element_located((By.XPATH, article_content_xpath))) 
    # + wait random time drawn from specific (strongly right-side-skewed) distribution to better imitate human behavior
    time.sleep(np.random.chisquare(1)+3)

    content = driver.find_element("xpath", article_content_xpath) # finds the content
    texts.append(content.text)
    
driver.close() # this closes the webdriver

print(len(texts))

end = time.time()
print(end-start)

30
476.46467566490173


### Example: UserAgent rotation

The User-Agent header is an HTTP header designed to identify the user agent responsible for executing a request. One can identify a scraper by it. It is possible to keep it changing. 

In [17]:
service = Service(executable_path = chromepath) 
options = webdriver.ChromeOptions(); options.add_argument("--headless")
driver = webdriver.Chrome(service = service, options = options)

driver.get("https://us.cnn.com/")

# returns our current User-Agent
user_agent = driver.execute_script("return navigator.userAgent;")
print("Current User-Agent:", user_agent)

driver.quit()

Current User-Agent: Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) HeadlessChrome/145.0.0.0 Safari/537.36


In [21]:
from fake_useragent import UserAgent

# generating fake (random) User-Agent
ua = UserAgent()
user_agent = ua.random

service = Service(executable_path = chromepath) 
options = webdriver.ChromeOptions()
options.add_argument("--headless")
options.add_argument(f"user-agent={user_agent}")
driver = webdriver.Chrome(service = service, options = options)

driver.get("https://us.cnn.com/")

user_agent = driver.execute_script("return navigator.userAgent;")
print("Current User-Agent:", user_agent)

driver.quit()

Current User-Agent: Mozilla/5.0 (Linux; Android 10; K) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/134.0.0.0 Mobile Safari/537.36


### Example: device ID rotation

One can be also identified and banned based on device ID. It is possible to emulate that we use a fake devide.

In [22]:
service = Service(executable_path = chromepath) 
options = webdriver.ChromeOptions(); options.add_argument("--headless")
options.add_experimental_option("mobileEmulation", {"deviceName": "Galaxy S5"})
driver = webdriver.Chrome(service = service, options = options)

driver.get("https://us.cnn.com/")

# returns our current User-Agent
user_agent = driver.execute_script("return navigator.userAgent;")
print("Current User-Agent:", user_agent)

driver.quit()

Current User-Agent: Mozilla/5.0 (Linux; Android 5.0; SM-G900P Build/LRX21T) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/145.0.0.0 Mobile Safari/537.36


List of available devices: https://github.com/DevExpress/device-specs/blob/master/devices.md.

### Example: IP rotation

In [23]:
service = Service(executable_path = chromepath) 

options = webdriver.ChromeOptions()
options.add_argument("--headless")

driver = webdriver.Chrome(service = service, options = options) # opens the browser

driver.get("https://sslproxies.org/") # webpage with some example IPs

xpath = "/html[1]/body[1]/section[1]/div[1]/div[2]/div[1]/table[1]/tbody[1]/tr[1]/td[1]"

WebDriverWait(driver, 30).until(EC.visibility_of_element_located((By.XPATH, xpath)))
time.sleep(np.random.chisquare(1)+3)

i = 1; proxy_list = []
while(True):
    try: # try getting all IPs you can one by one
        element1 = driver.find_element("xpath", "/html[1]/body[1]/section[1]/div[1]/div[2]/div[1]/table[1]/tbody[1]/tr["+str(i)+"]/td[1]")
        element2 = driver.find_element("xpath", "/html[1]/body[1]/section[1]/div[1]/div[2]/div[1]/table[1]/tbody[1]/tr["+str(i)+"]/td[2]")
        
        proxy_list.append(element1.text+":"+element2.text) # formates to "IP:port"
        
        i += 1 # if no error, then continue
    except: # error means no more rows in table with IPs, then break
        break
        
driver.close()

In [24]:
' '.join(proxy_list)

'38.145.218.163:8443 38.34.179.63:8443 38.34.179.71:8443 8.212.177.126:8080 57.128.188.167:9157 38.145.220.41:8443 38.145.203.86:8443 38.145.203.135:8443 38.145.208.229:8443 38.145.208.219:8443 38.145.208.228:8443 38.145.208.221:8443 38.145.208.223:8443 38.145.203.110:8443 38.145.208.239:8443 45.136.131.28:8447 38.34.179.38:8447 38.34.179.26:8450 85.198.96.242:3128 38.145.218.82:8443 45.136.130.194:8448 38.34.179.103:8443 38.34.179.46:8443 38.34.179.30:8443 38.34.179.91:8443 38.34.179.66:8443 38.34.179.89:8443 38.34.179.64:8443 45.136.131.41:8448 38.34.179.67:8443 45.136.130.177:8448 15.188.75.223:3128 83.219.250.8:62920 45.136.130.246:8443 45.136.130.185:8443 45.136.130.174:8447 38.34.179.14:8450 38.145.203.19:8447 38.34.179.28:8443 38.145.203.32:8443 38.34.179.33:8443 45.136.130.248:8443 38.145.218.51:8443 38.145.220.55:8443 45.136.131.47:8447 38.145.203.39:8443 38.145.203.111:8443 45.136.131.49:8447 165.227.5.10:8888 38.145.203.105:8443 38.145.220.103:8443 38.145.203.87:8443 193.23.

In [25]:
len(proxy_list)

100

In [29]:
from selenium.webdriver.common.proxy import Proxy, ProxyType

current_proxy_index = 0

def get_next_proxy():
    global current_proxy_index, proxy_list
    
    proxy = proxy_list[current_proxy_index]
    current_proxy_index = (current_proxy_index + 1) % len(proxy_list)
    # increases proxy index by 1 and takes the remains from division with the number of proxies
    # terefore it goes: 0, 1, 2, ..., 98, 99, 0, 1, 2, ...
    
    return proxy

def setup_driver(proxy_address):

    options = webdriver.ChromeOptions()

    options.add_argument("--headless")
    options.add_argument(f"--proxy-server=http://{proxy_address}")

    driver = webdriver.Chrome(options=options)

    return driver

Let us try using a proxy. We will start from https://api64.ipify.org to check if our IP changes.

In [30]:
service = Service(executable_path = chromepath) 

options = webdriver.ChromeOptions()
options.add_argument("--headless")

driver = webdriver.Firefox(service = service, options = options) # opens the browser

driver.get("https://api64.ipify.org")

WebDriverWait(driver, 30).until(EC.visibility_of_element_located((By.XPATH, "/html[1]/body[1]/pre[1]")))
time.sleep(np.random.chisquare(1)+3)

element = driver.find_element('xpath',"/html[1]/body[1]/pre[1]")
print("Current IP:", element.text)

driver.close()

Current IP: 194.9.78.5


In [32]:
# for i in range(100): 
# # we try it as many times as needed for https://api64.ipify.org to be accessed
# # neccessary as Internet connection can be unstable, and IP can be in use

#     print(i)
#     try:
#         proxy_address = get_next_proxy()
#         driver = setup_driver(proxy_address)
        
#         driver.get('https://api64.ipify.org')
        
#         WebDriverWait(driver, 30).until(EC.visibility_of_element_located((By.XPATH, "/html[1]/body[1]/pre[1]")))
#         time.sleep(np.random.chisquare(1)+3)

#         element = driver.find_element('xpath',"/html[1]/body[1]/pre[1]")
#         print("Current IP:", element.text)

#         driver.close()
#         break
        
#     except:
#         driver.close()
#         continue

It is not efficient as when we try certain IP, probably some other people are using them (as these are freely available online). To make the procedure more efficient, one should introduce a mechanism of building a machine with new IP, e.g. with **scrapoxy**.

### Supplementary materials

More Selenium options can be found here: https://www.seleniumeasy.com/.

### Exercise

Collect texts of court cases issued before the District Court in Warsaw in 01.01.2020-31.12.2024 using https://orzeczenia.ms.gov.pl/. Fill in the search tool as presented in `img/example.png` and search for the cases (i.e. click the 'Szukaj' button). You can fill certain parts of the website with content executing `element = driver.find_element(); element.send_keys()` Next, access the subpages with texts of court cases and collect them. Specify the delays wisely as this website involves CAPTCHAs.